# Lab 3 — Make Retrieval Better, and Prove It

This notebook builds on the Lab 2 RAG pipeline over the same `knowledge_base.json`.

**What this notebook does:**
1. Rebuilds a simple dense-retrieval baseline (Lab 2 style).
2. Adds a **hybrid search** upgrade: dense vector search + BM25 keyword search, merged with Reciprocal Rank Fusion (RRF).
3. Builds a 5-question evaluation set (with one exact-term question: `0x80070005`).
4. Scores both setups on two metrics:
   - **Retrieval hit rate** — exact check, no LLM needed.
   - **Faithfulness** — LLM-as-judge (yes/no), checks if the generated answer is fully supported by the retrieved context.
5. Writes the comparison table + conclusion to `eval_results.md`.

> Run this with `GOOGLE_API_KEY` exported in your environment and `pip install -r requirements.txt` done first.

## 0. Setup

In [ ]:
import sys
!{sys.executable} -m pip install -r requirements.txt


In [ ]:
import time

def generate(prompt, max_retries=6):
    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(model=GEN_MODEL, contents=prompt)
            return resp.text.strip()
        except Exception as e:
            if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
                wait = 15 * (attempt + 1)
                print(f"Rate limited, waiting {wait}s... (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Failed after retries due to rate limiting.")

In [ ]:
import os, json, math
import numpy as np
from rank_bm25 import BM25Okapi
from google import genai


assert os.environ.get("GOOGLE_API_KEY"), "Run: $env:GOOGLE_API_KEY= before launching Jupyter, or load it from .env"

assert os.environ.get("GOOGLE_API_KEY")
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

EMBED_MODEL = "models/gemini-embedding-001"
GEN_MODEL = "models/gemini-flash-lite-latest"


## 1. Load the knowledge base

In [ ]:
with open("knowledge_base.json") as f:
    KB = json.load(f)

KB_BY_ID = {doc["id"]: doc for doc in KB}
print(f"Loaded {len(KB)} passages")
for d in KB:
    print(d["id"], "-", d["text"][:60], "...")


## 2. Embedding + dense retrieval (Lab 2 baseline)

This is the same dense-vector approach from Lab 2: embed every passage once, embed the query at search time, rank by cosine similarity.

In [ ]:
def embed_text(text, task_type="RETRIEVAL_DOCUMENT", max_retries=6):
    for attempt in range(max_retries):
        try:
            result = client.models.embed_content(model=EMBED_MODEL, contents=text)
            return np.array(result.embeddings[0].values, dtype=np.float32)
        except Exception as e:
            if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
                wait = 15 * (attempt + 1)
                print(f"Rate limited (embed), waiting {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Embedding failed after retries due to rate limiting.")

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

In [ ]:
def embed_text(text, task_type="RETRIEVAL_DOCUMENT"):
    """Get a single embedding vector from Gemini."""
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=text,
    )
    return np.array(result.embeddings[0].values, dtype=np.float32)

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

# Pre-compute (and cache) embeddings for every passage once
print("Embedding knowledge base passages...")
KB_EMBEDDINGS = {doc["id"]: embed_text(doc["text"]) for doc in KB}
print("Done.")


In [ ]:
for m in client.models.list():
    if "embedContent" in m.supported_actions:
        print(m.name)

In [ ]:
def retrieve_dense(query, k=3):
    """Lab 2 baseline: pure dense vector search, top-k by cosine similarity."""
    q_vec = embed_text(query, task_type="RETRIEVAL_QUERY")
    scored = [(doc_id, cosine_sim(q_vec, vec)) for doc_id, vec in KB_EMBEDDINGS.items()]
    scored.sort(key=lambda x: -x[1])
    top_ids = [doc_id for doc_id, _ in scored[:k]]
    return [KB_BY_ID[i] for i in top_ids]

# quick smoke test
for d in retrieve_dense("How do I get a refund?", k=3):
    print(d["id"], "-", d["text"][:60])


## 3. Upgrade: Hybrid search (dense + BM25, merged with RRF)

Dense embeddings are great at *meaning* but can miss exact tokens like error codes (`0x80070005`), since those tokens
get diluted in a semantic vector. BM25 is the opposite: great at exact keyword/term matches, weaker at paraphrase.

We retrieve a wider candidate pool (top 8) from each method, then merge the two rankings with
**Reciprocal Rank Fusion (RRF)** — a standard, scale-free way to combine rankings without needing to normalize
two different score distributions (cosine similarity vs. BM25 score).

In [ ]:
# Tokenize the corpus once for BM25 (simple lowercase whitespace split is enough here)
CORPUS_IDS = [doc["id"] for doc in KB]
CORPUS_TOKENS = [doc["text"].lower().split() for doc in KB]
bm25 = BM25Okapi(CORPUS_TOKENS)

def retrieve_bm25(query, k=8):
    scores = bm25.get_scores(query.lower().split())
    ranked = sorted(zip(CORPUS_IDS, scores), key=lambda x: -x[1])
    top_ids = [doc_id for doc_id, _ in ranked[:k]]
    return [KB_BY_ID[i] for i in top_ids]

# quick smoke test on the exact-term case
for d in retrieve_bm25("0x80070005", k=3):
    print(d["id"], "-", d["text"][:60])


In [ ]:
def reciprocal_rank_fusion(rankings, rrf_k=60):
    """rankings: list of ordered id-lists (best first). Returns ids sorted by fused score."""
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (rrf_k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])

def retrieve_hybrid(query, top_k=3, pool_k=8):
    dense_ids = [d["id"] for d in retrieve_dense(query, k=pool_k)]
    bm25_ids = [d["id"] for d in retrieve_bm25(query, k=pool_k)]
    fused = reciprocal_rank_fusion([dense_ids, bm25_ids])
    top_ids = [doc_id for doc_id, _ in fused[:top_k]]
    return [KB_BY_ID[i] for i in top_ids]

# quick smoke test
for d in retrieve_hybrid("0x80070005", top_k=3):
    print(d["id"], "-", d["text"][:60])


## 4. Generation: answer a question from retrieved context

In [ ]:
def generate(prompt, max_retries=6):
    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(model=GEN_MODEL, contents=prompt)
            return resp.text.strip()
        except Exception as e:
            if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
                wait = 15 * (attempt + 1)
                print(f"Rate limited, waiting {wait}s... (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Failed after retries due to rate limiting.")

def generate_answer(query, retrieved_docs):
    context = "\n\n".join(f"[{d['id']}] {d['text']}" for d in retrieved_docs)
    prompt = f"""Answer the question using ONLY the context below. If the context does not contain
the answer, say you don't have enough information.

Context:
{context}

Question: {query}

Answer:"""
    return generate(prompt)

In [ ]:
def generate(prompt):
    resp = client.models.generate_content(model=GEN_MODEL, contents=prompt)
    return resp.text.strip()

def generate_answer(query, retrieved_docs):
    context = "\n\n".join(f"[{d['id']}] {d['text']}" for d in retrieved_docs)
    prompt = f"""Answer the question using ONLY the context below. If the context does not contain
the answer, say you don't have enough information.

Context:
{context}

Question: {query}

Answer:"""
    return generate(prompt)


## 5. Faithfulness — LLM-as-judge

For each generated answer, ask the model whether the answer is fully supported by the retrieved context
(no unsupported / hallucinated claims). The judge only sees the retrieved context and the answer — not the full KB —
so it mirrors what a real faithfulness check should verify.

In [ ]:
def judge_faithfulness(answer, retrieved_docs):
    context = "\n\n".join(d["text"] for d in retrieved_docs)
    prompt = f"""You are a strict evaluator. Given the CONTEXT and the ANSWER, decide whether every claim in
the ANSWER is fully supported by the CONTEXT. If the answer adds information not present in the context,
or contradicts it, that counts as NOT supported.

Respond with exactly one word: YES or NO.

CONTEXT:
{context}

ANSWER:
{answer}

Verdict (YES or NO):"""
    verdict = generate(prompt).strip().upper()
    return verdict.startswith("Y")


## 6. Evaluation set (5 questions)

Each question has the `id` of the passage that *should* be retrieved. Question 1 is the exact-term case
(`0x80070005`) that plain dense retrieval tends to fumble, since the error code is just a token, not a concept
a dense embedding model represents well.

In [ ]:
EVAL_SET = [
    {"q": "What does error code 0x80070005 mean and how do I fix it?", "expected_id": "kb-08"},
    {"q": "How many days do I have to get a full refund after buying something?", "expected_id": "kb-04"},
    {"q": "How do I reset my account password?", "expected_id": "kb-07"},
    {"q": "When are employees allowed to park in lot B?", "expected_id": "kb-01"},
    {"q": "How often does the office kitchen get restocked?", "expected_id": "kb-10"},
]
len(EVAL_SET)


## 7. Run the evaluation for one retrieval function

In [ ]:
def run_eval(retrieve_fn, label):
    rows = []
    hits = 0
    faithful = 0
    for item in EVAL_SET:
        retrieved = retrieve_fn(item["q"])
        retrieved_ids = [d["id"] for d in retrieved]
        hit = item["expected_id"] in retrieved_ids
        hits += int(hit)

        answer = generate_answer(item["q"], retrieved)
        is_faithful = judge_faithfulness(answer, retrieved)
        faithful += int(is_faithful)

        rows.append({
            "question": item["q"],
            "expected_id": item["expected_id"],
            "retrieved_ids": retrieved_ids,
            "hit": hit,
            "answer": answer,
            "faithful": is_faithful,
        })
        print(f"[{label}] hit={hit} faithful={is_faithful} | {item['q'][:50]}")
        time.sleep(4)
    hit_rate = hits / len(EVAL_SET)
    faithfulness_rate = faithful / len(EVAL_SET)
    return hit_rate, faithfulness_rate, rows

    hit_rate = hits / len(EVAL_SET)
    faithfulness_rate = faithful / len(EVAL_SET)
    return hit_rate, faithfulness_rate, rows


In [ ]:
for m in client.models.list():
    if "generateContent" in m.supported_actions:
        print(m.name)

In [ ]:
print("=== Baseline (dense only, top-3) ===")
baseline_hit_rate, baseline_faith_rate, baseline_rows = run_eval(
    lambda q: retrieve_dense(q, k=3), "baseline"
)


In [ ]:
print("=== Upgraded (hybrid: dense + BM25 via RRF, top-3 of pool-8) ===")
hybrid_hit_rate, hybrid_faith_rate, hybrid_rows = run_eval(
    lambda q: retrieve_hybrid(q, top_k=3, pool_k=8), "hybrid"
)


## 8. Comparison table + conclusion

In [ ]:
print("| Setup | Retrieval Hit Rate | Faithfulness |")
print("|---|---|---|")
print(f"| Baseline (dense only) | {baseline_hit_rate:.0%} | {baseline_faith_rate:.0%} |")
print(f"| Upgraded (hybrid: dense + BM25, RRF) | {hybrid_hit_rate:.0%} | {hybrid_faith_rate:.0%} |")


In [ ]:
# Build a data-driven conclusion paragraph from the actual numbers above.

def fmt_pct(x):
    return f"{x:.0%}"

hit_delta = hybrid_hit_rate - baseline_hit_rate
faith_delta = hybrid_faith_rate - baseline_faith_rate

# Did hybrid specifically fix the exact-term (error code) question?
exact_term_row_baseline = baseline_rows[0]   # EVAL_SET[0] is the 0x80070005 question
exact_term_row_hybrid = hybrid_rows[0]

if hit_delta > 0:
    hit_verdict = "helped"
elif hit_delta < 0:
    hit_verdict = "hurt"
else:
    hit_verdict = "made no difference to"

exact_term_note = ""
if not exact_term_row_baseline["hit"] and exact_term_row_hybrid["hit"]:
    exact_term_note = (
        " Notably, the baseline missed the exact-term question (`0x80070005`), while the hybrid setup "
        "retrieved the correct passage — exactly the failure mode hybrid search is meant to fix, since BM25 "
        "catches exact tokens that dense embeddings can dilute."
    )
elif exact_term_row_baseline["hit"] and exact_term_row_hybrid["hit"]:
    exact_term_note = " Both setups retrieved the correct passage for the exact-term question."
elif not exact_term_row_baseline["hit"] and not exact_term_row_hybrid["hit"]:
    exact_term_note = " Neither setup retrieved the correct passage for the exact-term question — worth a closer look at the RRF weighting or pool size."

conclusion = (
    f"Hybrid search (dense + BM25, RRF) {hit_verdict} retrieval hit rate, moving it from "
    f"{fmt_pct(baseline_hit_rate)} to {fmt_pct(hybrid_hit_rate)} ({hit_delta:+.0%}).{exact_term_note} "
    f"Faithfulness moved from {fmt_pct(baseline_faith_rate)} to {fmt_pct(hybrid_faith_rate)} "
    f"({faith_delta:+.0%}), which is expected since faithfulness mostly tracks whether the right passage "
    f"was in context in the first place — better retrieval gives the generator better material to be faithful to."
)
print(conclusion)


## 9. (Optional stretch) Query rewriting as a third column

In [ ]:
def rewrite_query(query):
    prompt = f"""Rewrite the following short question into a more explicit search query for a keyword+vector
retrieval system. Expand abbreviations, add likely synonyms, keep it to one sentence. Return ONLY the rewritten
query, nothing else.

Question: {query}
Rewritten query:"""
    return generate(prompt)

def retrieve_hybrid_with_rewrite(query, top_k=3, pool_k=8):
    expanded = rewrite_query(query)
    return retrieve_hybrid(expanded, top_k=top_k, pool_k=pool_k)

print("=== Hybrid + query rewriting ===")
rewrite_hit_rate, rewrite_faith_rate, rewrite_rows = run_eval(
    retrieve_hybrid_with_rewrite, "hybrid+rewrite"
)


## 10. Write `eval_results.md`

In [ ]:
include_rewrite = "rewrite_hit_rate" in dir()

table_rows = [
    ("Baseline (dense only)", baseline_hit_rate, baseline_faith_rate),
    ("Upgraded (hybrid: dense + BM25, RRF)", hybrid_hit_rate, hybrid_faith_rate),
]
if include_rewrite:
    table_rows.append(("Hybrid + query rewriting (stretch)", rewrite_hit_rate, rewrite_faith_rate))

table_md = "| Setup | Retrieval Hit Rate | Faithfulness |\n|---|---|---|\n"
for label, hr, fr in table_rows:
    table_md += f"| {label} | {hr:.0%} | {fr:.0%} |\n"

report = f"""# Eval Results — Lab 3 (Hybrid Search vs. Baseline)

## Eval set
5 questions over `knowledge_base.json`, each with an expected passage id. One question
(`0x80070005` error code) targets the exact-term failure mode of pure dense retrieval.

## Comparison table

{table_md}
## Conclusion

{conclusion}
"""

with open("eval_results.md", "w") as f:
    f.write(report)

print(report)


## Notes for the README submission

- `retrieve_dense` is the Lab 2 baseline (pure cosine similarity over Gemini embeddings).
- `retrieve_hybrid` is the Lab 3 upgrade: BM25 (keyword) + dense (semantic), merged with Reciprocal Rank Fusion —
  no manual score normalization needed, and it is robust to BM25/cosine living on different scales.
- Hit rate is a deterministic check (`expected_id in retrieved_ids`) — no LLM involved, so it is not affected by
  judge variance.
- Faithfulness uses the model itself as judge (yes/no), scoped only to the retrieved context, not the full KB.
- `eval_results.md` is regenerated every run from the live numbers above — nothing here is hand-typed, so the
  conclusion always matches whatever the model actually produced.